# Unit 2: RAG, Vector Stores, and Indexing

## Introduction
LLMs have a knowledge cutoff and can hallucinate. **Retrieval Augmented Generation (RAG)** solves this by retrieving relevant data and injecting it into the prompt.

In this notebook, we will master:
1.  **Embeddings:** Representing text as vectors.
2.  **Vector Stores:** Storing and searching vectors (FAISS).
3.  **Naïve RAG:** The standard Retrieval -> Augment -> Generate pipeline.
4.  **Indexing Challenges:** Deep dive into how vector databases search efficiently (Flat, IVF, HNSW, PQ).

---

## Part 4a: Embeddings & Vector Space

### 1. Introduction: Computers Don't Read English

If you ask a computer "Is a cat similar to a dog?", it doesn't know. To a computer, "cat" is just a string of bytes: `01100011...`.

To solve this, we use **Embeddings**.

### What is an Embedding?
An embedding is a translation from **Words** to **Lists of Numbers (Vectors)**, such that similar words represent close numbers.

### The Process (Flowchart)
```mermaid
graph LR
    A["Input Text ('Apple')"] -->|Tokenization| B["Tokens (101, 255)"]
    B -->|Embedding Model| C["Vector List ([0.1, -0.5, 0.9...])"]
    C -->|Store| D["Vector Database"]
```

In [2]:
!pip install -q \
transformers==4.37.2 \
sentence-transformers==2.6.1 \
langchain==0.1.20 \
langchain-community==0.0.38 \
langchain-huggingface==0.0.3 \
faiss-cpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 98.6 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 63.4 kB/s eta 0:00:00
ERROR: Cannot install langchain-huggingface==0.0.3 and transformers==4.37.2 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts


In [5]:
!pip install -q langchain-huggingface

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector = embeddings.embed_query("Apple")

print("Dim:", len(vector))
print("First 5:", vector[:5])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Dim: 384
First 5: [-0.006138487718999386, 0.03101177327334881, 0.06479360908269882, 0.01094149798154831, 0.005267191678285599]


In [7]:

%pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

vector = model.encode("Apple")

print("Dim:", len(vector))
print("First 5:", vector[:5])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Dim: 384
First 5: [-0.00613849  0.03101177  0.06479361  0.0109415   0.00526719]


## 2. Viewing a Vector

Let's see what the word "Apple" looks like to the machine.

### Conceptual Note: Dimensions
The vector below has **384 dimensions** (for MiniLM).
- Imagine a graph with X and Y axes (2 Dimensions). You can plot a point (x, y).
- Now imagine adding Z (3 Dimensions).
- Now imagine **384 axes**.

Each axis represents a feature (e.g., "Is it a fruit?", "Is it red?", "Is it tech-related?"). The numbers aren't random; they encode meaning.

In [8]:
vector = model.encode("Apple")

print(f"Dimensionality: {len(vector)}")
print(f"First 5 numbers: {vector[:5]}")

Dimensionality: 384
First 5 numbers: [-0.00613849  0.03101177  0.06479361  0.0109415   0.00526719]


## 3. The Math: Cosine Similarity

How do we know if two vectors are close? We measure the **Angle** between them.

### Cosine Similarity Formula
$$ \text{Similarity} = \cos(\theta) = \frac{A \cdot B}{\|A\| \|B\|} $$

- **1.0**: Arrows point in the Exact Same Direction (Identical).
- **0.0**: Arrows are Perpendicular (Unrelated).
- **-1.0**: Arrows point in Opposite Directions (Opposite).

**Experiment:** Let's compare "Cat", "Dog", and "Car".

In [9]:
import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# ✅ use model.encode instead
vec_cat = model.encode("Cat")
vec_dog = model.encode("Dog")
vec_car = model.encode("Car")

print(f"Cat vs Dog: {cosine_similarity(vec_cat, vec_dog):.4f}")
print(f"Cat vs Car: {cosine_similarity(vec_cat, vec_car):.4f}")

Cat vs Dog: 0.6606
Cat vs Car: 0.4633


### Analysis
You should see that **Cat & Dog** score higher (e.g., ~0.8) than **Cat & Car** (e.g., ~0.3).
This Mathematical Distance is the foundation of all Search engines and RAG systems.

This is arguably the most important concept in modern AI.

# Unit 2 - Part 4b: Naive RAG Pipeline

## 1. Introduction: The Open-Book Test

RAG (Retrieval-Augmented Generation) is just an Open-Book Test architecture.
1.  **Retrieval:** Find the right page in the textbook.
2.  **Generation:** Write the answer using that page.

### The Pipeline (Flowchart)
```mermaid
graph TD
    User[User Question] --> Retriever[Retriever System]
    Retriever -->|Search Database| Docs[Relevant Documents]
    Docs --> Combiner[Prompt Template]
    User --> Combiner
    Combiner -->|Full Prompt w/ Context| LLM[Gemini Model]
    LLM --> Answer[Final Answer]
```

In [10]:
%pip install -q faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 47.8 MB/s eta 0:00:00


In [11]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
documents = [
    "Python is used for AI and machine learning.",
    "FAISS is used for fast similarity search.",
    "RAG stands for Retrieval Augmented Generation.",
    "Embeddings convert text into vectors."
]

In [13]:
import numpy as np

doc_vectors = model.encode(documents)

In [14]:
import faiss

dimension = doc_vectors.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(doc_vectors)

In [15]:
query = "What is FAISS?"

query_vector = model.encode([query])

distances, indices = index.search(query_vector, k=2)

print("Top results:")
for i in indices[0]:
    print("-", documents[i])

Top results:
- FAISS is used for fast similarity search.
- Python is used for AI and machine learning.


## 2. The "Knowledge Base" (Grounding)

LLMs hallucinate because they rely on "parametric memory" (what they learned during training).
RAG introduces "non-parametric memory" (external facts).

Let's define some facts the LLM definitely *does not* know.

In [16]:
from langchain_core.documents import Document

docs = [
    Document(page_content="Neeharika's favorite food is Fries with extra spice."),
    Document(page_content="The secret password to the lab is 'Blueberry'."),
    Document(page_content="LangChain is a framework for developing applications powered by language models."),
]

## 3. Indexing ( Storing the knowledge)

We use **FAISS** (Facebook AI Similarity Search) to store the embeddings.
Think of FAISS as a super-fast librarian that organizes books by content, not title.

In [17]:
doc_texts = [doc.page_content for doc in docs]

doc_vectors = model.encode(doc_texts)

In [18]:
import faiss
import numpy as np

dimension = doc_vectors.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(doc_vectors))

In [19]:
def retrieve(query, k=2):
    query_vector = model.encode([query])
    distances, indices = index.search(np.array(query_vector), k)

    results = [doc_texts[i] for i in indices[0]]
    return results

## 4. The RAG Chain

We use LCEL to stitch it together.

**Step 1:** The `retriever` takes the question, converts it to numbers, and finds the closest document.
**Step 2:** `RunnablePassthrough` holds the question.
**Step 3:** The `prompt` combines them.

In [20]:
query = "What is LangChain?"

results = retrieve(query)

print("Retrieved docs:")
for r in results:
    print("-", r)

Retrieved docs:
- LangChain is a framework for developing applications powered by language models.
- Neeharika's favorite food is Fries with extra spice.


In [21]:
query = "What is the secret password?"

retrieved_docs = retrieve(query)

In [22]:
context = "\n".join(retrieved_docs)

In [23]:
prompt = f"""
Answer based ONLY on the context below:

{context}

Question: {query}
"""

In [24]:
# simple "LLM" style response
print("----- Retrieved Context -----")
print(context)

print("\n----- Final Answer -----")

# simulate answer extraction
for doc in retrieved_docs:
    if "password" in doc.lower():
        print(doc)

----- Retrieved Context -----
The secret password to the lab is 'Blueberry'.
LangChain is a framework for developing applications powered by language models.

----- Final Answer -----
The secret password to the lab is 'Blueberry'.


## 5. Analysis

The retrieval step is opaque here. In the next notebook (**4c**), we will look *inside* the retriever to understand how FAISS actually finds that document among millions of others.

# Unit 2 - Part 4c: Deep Dive into Indexing Algorithms

## 1. Introduction: The Scale Problem

Comparing 1 vector against 10 vectors is fast.
Comparing 1 vector against **100 Million** vectors is slow.

**FAISS (Facebook AI Similarity Search)** was built to solve this.

### The Trade-off Triangle
You can pick 2:
- **Speed** (Query time)
- **Accuracy** (Recall)
- **Memory** (RAM usage)

We will explore algorithms that optimize different corners of this triangle.

In [25]:
import faiss
import numpy as np

# Mock Data: 10,000 vectors of size 128
d = 128
nb = 10000
xb = np.random.random((nb, d)).astype('float32')

## 2. Flat Index (Brute Force)

**Concept:** Check every single item.

- **Algo:** `IndexFlatL2`
- **Pros:** 100% Accuracy (Gold Standard).
- **Cons:** Slow (O(N)). Unusable at 1M+ vectors.


In [26]:
index = faiss.IndexFlatL2(d)
index.add(xb)
print(f"Flat Index contains {index.ntotal} vectors")

Flat Index contains 10000 vectors


## 3. IVF (Inverted File Index)

**Concept:** Clustering / Partitioning.

Imagine looking for a book. Instead of checking every shelf, you go to the "Sci-Fi" section. Then you only search books *in that section*.

### How it works (Flowchart)
```mermaid
graph TD
    Data[All 1M Vectors] -->|Train| Clusters[1000 Cluster Centers (Centroids)]
    Query[User Query] -->|Step 1| FindClosest[Find Closest Centroid]
    FindClosest -->|Step 2| Search[Search ONLY vectors in that Cluster]
```

**Analogy:** Voronoi Cells (Zip Codes). We only search the local zip code.

In [27]:
nlist = 100 # How many 'zip codes' (clusters) we want
quantizer = faiss.IndexFlatL2(d) # The calculator for distance
index_ivf = faiss.IndexIVFFlat(quantizer, d, nlist)

# We MUST train it first so it learns where the clusters are
index_ivf.train(xb)
index_ivf.add(xb)

## 4. HNSW (Hierarchical Navigable Small World)

**Concept:** Six Degrees of Separation.

Most data is connected. HNSW builds a **Graph**.
- **Layer 0:** Every point connects to neighbors.
- **Layer 1:** "Express Highways" connecting distant points.

**Analogy:** Catching a flight.
You don't fly Local -> Local -> Local.
You fly Local -> **HUB** (Chicago) -> **HUB** (London) -> Local.

- **Pros:** Extremely fast retrieval.
- **Cons:** Heavier on RAM (needs to store the edges of the graph).

In [28]:
M = 16 # Number of connections per node (The 'Hub' factor)
index_hnsw = faiss.IndexHNSWFlat(d, M)
index_hnsw.add(xb)

## 5. PQ (Product Quantization)

**Concept:** Compression (Lossy).

Do we need 32-bit float precision (`0.123456789`)? No. `0.12` is fine.
PQ breaks the vector into chunks and approximates them.

**Analogy:** 4K Video vs 480p Video.
- 480p is blurry, but it's 10x smaller and faster to stream.
- Use PQ when you are RAM constrained (e.g., storing 1 Billion vectors).

In [29]:
m = 8 # Split vector into 8 sub-vectors
index_pq = faiss.IndexPQ(d, m, 8)
index_pq.train(xb)
index_pq.add(xb)
print("PQ Compression complete. RAM usage minimized.")

PQ Compression complete. RAM usage minimized.
